# **Dataset e Bibliotecas**

 ## Pré

In [ ]:

import torch
from torch import nn
import torch.nn.functional as F
from torch.utils.data import DataLoader,Subset, Dataset, ConcatDataset
from torchvision import datasets, transforms
from torchvision.transforms.functional import to_tensor
from torchsummary import summary

import numpy as np
import matplotlib.pyplot as plt
import cv2

from sklearn.model_selection import train_test_split

import random
from tqdm.auto import tqdm
import os
import requests
import zipfile
from pathlib import Path
from PIL import Image
import json
from datetime import datetime
import re

# Agnostico
device = "cuda" if torch.cuda.is_available() else "cpu"

# Checa a versão Pytorch
torch.__version__

In [ ]:
!nvidia-smi

## Dataset

In [ ]:
# Caminho dos Datasets
path_data = Path("data")

# Caminho do Dataset

#dataset_dir = path_data / "dataset_V00"
water_bottle_dataset = path_data / "pre_dataset/water_bottle_dataset"
dataset_50_samples = path_data / "dataset_50_samples/water_bottle_dataset"
dataset_no_frames = path_data / "dataset_no_frames/water_bottle_dataset"
dataset_50_no_frame = path_data / "grasp_dataset_50_no_frames/dataset/water_bottle_dataset"

dataset_dir = dataset_50_no_frame

# Tamanhos dos batches
batch_size = 2

In [ ]:

# Verifica existência do diretório
if dataset_dir.is_dir():

    views_count = sum(1 for p in dataset_dir.iterdir() if p.is_dir())-1

    print(f"{dataset_dir} diretório existe com {views_count} vistas.")

    random_view_id = random.randint(0,views_count)
    random_view_dir = dataset_dir / f"view_{random_view_id}"

    random_rgb = np.copy(np.load(random_view_dir / "rgb.npy", mmap_mode="r"))
    random_normal_map = np.copy(np.load(random_view_dir / "normal.npy", mmap_mode="r"))

    img_shape = np.copy(np.load(random_view_dir / "depth.npy", mmap_mode="r")).shape

    print(f"\n---------------------------------- Dados da vista {random_view_id} ----------------------------------\n")
    print(f"View: view_{random_view_id:02d} | Image shape: {img_shape}")

    fig, ax = plt.subplots(1, 2, figsize=(12, 7))

    ax[0].imshow(random_rgb)
    ax[0].axis(False);

    ax[1].imshow(random_normal_map)
    ax[1].axis(False);

else:
    print(f"Did not find {dataset_dir} directory")

In [ ]:
def preprocess_data(rgb, depth, normal, mask, target):

    mask = cv2.inRange(mask, 1, 1).astype(bool).astype(int) # Seleciona apenas o objeto, retirando o gripper ou outros elementos

    depth[mask == 0] = 2 # Coloca tudo que não é o objeto a 2 metros de distância
    depth = depth/depth.max() # Normaliza a profundidade

    normal[mask == 0] = np.array([0.49803922, 0.49803922, 1. ])*255 # Retira normais indesejadas

    #grasp = (cv2.inRange(grasp, 0, 0).astype(bool) * mask).astype(int) + grasp # Preenche os espaços vazios da garrafa

    kernel = np.ones((2, 2), np.uint8)  # Definir o tamanho do kernel (ajuste conforme necessário)
    target = cv2.dilate(target, kernel, iterations=2)

    return rgb, depth, normal, mask, target

In [ ]:
class MultiViewObjectDataset(Dataset):
    """
    Cada pasta view_* contém só UM frame (rgb, depth, normal, mask).
    Assim, cada item do dataset = uma vista.
    """
    @staticmethod
    def _make_tensor(*, arr, channels_last=False, long=False):
        t = torch.from_numpy(arr)
        if channels_last:
            t = t.permute(2, 0, 1)
        if long:
            t = t.long()
        return t.contiguous()

    def __init__(self, root, *, mmap=True):
        self.root = root
        self.mmap = mmap
        self.img_size = img_shape
        # lista das pastas que começam por view_
        self.views = sorted(p for p in os.listdir(root) if p.startswith("view_"))


    def __len__(self):
        return len(self.views)        # = n_views

    def _load(self, path):
        mode = "r" if self.mmap else None
        dat = np.load(path, mmap_mode=mode)
        return np.copy(dat)

    def _load_taget_json(self, path):
        mode = "r" if self.mmap else None
        with open(path, mode) as arq:
            dados = json.load(arq)

        final_contacts = np.array(dados[0]['final_contacts'])
        random_y = np.array(dados[0]['random_y'])
        random_x = np.array(dados[0]['random_x'])
        json_data = np.column_stack((random_y, random_x, final_contacts))

        dat= np.ones(self.img_size) * (-1)

        for i in range(len(json_data)):
            sim_output = json_data[i][2]

            if(sim_output): sim_output=1
            else: sim_output=0

            dat[json_data[i][0]][json_data[i][1]] = sim_output

        return np.copy(dat)

    def __getitem__(self, idx):
        vname = self.views[idx]
        vdir  = os.path.join(self.root, vname)

        # Carregando dados
        rgb   = self._load(f"{vdir}/rgb.npy")               # (H,W,3)
        depth = self._load(f"{vdir}/depth.npy")     # (H,W)
        normal= self._load(f"{vdir}/normal.npy")        # (H,W,3)
        mask  = self._load(f"{vdir}/segmentation.npy")
        grasp = self._load_taget_json(f"{vdir}/metadata.json")        # ← alvo

        # Pré processamento
        rgb, depth, normal, mask, grasp = preprocess_data(rgb, depth, normal, mask, grasp)

        # Transformando para tensores
        rgb_t   = self._make_tensor(arr=rgb,   channels_last=True).float() / 255
        depth_t = self._make_tensor(arr=depth).unsqueeze(0)            # (1,H,W)
        normal_t= self._make_tensor(arr=normal, channels_last=True).float() / 255
        mask_t  = self._make_tensor(arr=mask,   long=True).unsqueeze(0)
        grasp_t = torch.from_numpy(grasp).float()

        return {"rgb": rgb_t, "depth": depth_t,
                "normal": normal_t, "mask": mask_t,
                "grasp": grasp_t, "view": vname}

dataset = ConcatDataset([
    MultiViewObjectDataset(water_bottle_dataset, mmap=True)
    ,MultiViewObjectDataset(dataset_50_samples, mmap=True)
    ,MultiViewObjectDataset(dataset_no_frames, mmap=True)
])

dataset = ConcatDataset([MultiViewObjectDataset(dataset_50_no_frame, mmap=True)])

views_count = dataset.cumulative_sizes[-1]-1
views_count

In [ ]:
n = 2

#n = random.randint(0,views_count)

rgb = dataset[n]["rgb"].permute(1,2,0).squeeze().numpy()
normal = dataset[n]["normal"].permute(1,2,0).squeeze().numpy()
target = dataset[n]["grasp"].numpy()
seg = cv2.inRange(dataset[n]["mask"].permute(1,2,0).squeeze().numpy(), 1, 1)
depth = dataset[n]["depth"].permute(1,2,0).squeeze().numpy()


plt.figure(figsize=(15,15))
plt.imshow(target)
plt.title(f"Ground Truth {n}")
plt.axis(False);

print(target)

In [ ]:
# --- gera lista de índices ---
all_idx = list(range(len(dataset)))

# opcional: se quiser estratificar por uma classe binária salva em grasp[0]
# y_strat = [dataset[i]["grasp"][0].item() for i in all_idx]
# train_idx, test_idx = train_test_split(all_idx, test_size=0.2,
#                                        random_state=42, stratify=y_strat)

train_idx, test_idx = train_test_split(
    all_idx, test_size=0.3, random_state=42, shuffle=True
)

train_set = Subset(dataset, train_idx)
test_set  = Subset(dataset, test_idx)


In [ ]:
train_loader = DataLoader(train_set, batch_size = batch_size, shuffle=True,
                          num_workers=4, pin_memory=True)

test_loader  = DataLoader(test_set,  batch_size = batch_size, shuffle=False,
                          num_workers=4, pin_memory=True)


## Funções

In [ ]:
def accuracy_fn(y_true, y_pred):
    """Calculates accuracy between truth labels and predictions.

    Args:
        y_true (torch.Tensor): Truth labels for predictions.
        y_pred (torch.Tensor): Predictions to be compared to predictions.

    Returns:
        [torch.float]: Accuracy value between y_true and y_pred, e.g. 78.45
    """
    correct = torch.eq(y_true, y_pred).sum().item()/batch_size
    acc = (correct / y_pred.shape.numel()) * 100
    return acc

In [ ]:
def print_cuda_memory():
    print(f"\n{torch.cuda.memory_allocated()/1e6} Mb de memória alocada")

In [ ]:
def plot_prediction(model:nn.Module, input:torch.Tensor, plot_data:dict, target:np.ndarray = np.zeros(img_shape), raw : bool = True, lim = 0.5):

    output = model(input).cpu().detach()

    #random_y = torch.softmax(random_y, dim=2)
    #random_y = torch.argmax(random_y, dim=0)

    output_np = output.squeeze().numpy()

    y_target, x_target = np.unravel_index(np.argmax(output_np), output_np.shape)
    y_targetSeg, x_targetSeg = np.unravel_index(np.argmax(output_np*plot_data["mask"]), output_np.shape)

    # Convertendo probabilidades para valores booleanos
    if(not(raw)):
        output_np = output_np >= lim

    print(f"\n-----------------------------------------------------------------------")

    fig, ax = plt.subplots(2, 3, figsize=(16, 8))
    plot1 = ax[0][0]
    plot2 = ax[1][0]
    plot3 = ax[0][1]
    plot4 = ax[1][1]
    plot5 = ax[0][2]
    plot6 = ax[1][2]

    plot1.imshow(plot_data["rgb"])
    plot1.set_title("RGB")
    plot1.axis(False);

    plot2.imshow(plot_data["normal"])
    plot2.set_title("Normal Map")
    plot2.axis(False);

    plot3.imshow(plot_data["depth"],cmap="gray")
    plot3.set_title("Depth")
    plot3.axis(False);

    plot4.imshow(plot_data["mask"],cmap="gray")
    plot4.set_title("Segmentation Mask")
    plot4.axis(False);

    plot5.imshow(target)
    plot5.set_title("Ground Truth")
    plot5.axis(False);

    plot6.imshow(output_np)
    plot6.plot(x_target, y_target, 'r+', markersize=3)
    plot6.plot(x_targetSeg, y_targetSeg, 'r^', markersize=3)
    plot6.set_title("Model Output")
    plot6.axis(False);

    return output_np

In [ ]:
normalize = transforms.Normalize(0.5, 0.5)

# Modelos, Treinamento e Teste

## Modelos

### Modelo 1

In [ ]:
# Rede gerada pelo chat GPT


# ---------- Blocos auxiliares ----------
def double_conv(c_in, c_out, *, groups=8):
    return nn.Sequential(
        nn.Conv2d(c_in, c_out, 3, padding=1, bias=False),
        nn.GroupNorm(groups, c_out),     # <— economiza no running-mean
        nn.ReLU(inplace=True),
        nn.Conv2d(c_out, c_out, 3, padding=1, bias=False),
        nn.GroupNorm(groups, c_out),
        nn.ReLU(inplace=True),
    )

class Down(nn.Module):
    def __init__(self, c_in, c_out):
        super().__init__()
        self.block = nn.Sequential(
            nn.MaxPool2d(2), double_conv(c_in, c_out)
        )
    def forward(self, x): return self.block(x)

class Up(nn.Module):
    def __init__(self, c_in, c_out):
        super().__init__()
        self.up = nn.ConvTranspose2d(c_in, c_in // 2, 2, stride=2)
        self.conv = double_conv(c_in, c_out)
    def forward(self, x1, x2):            # x1 = decoder, x2 = skip
        x1 = self.up(x1)
        x = torch.cat([x2, x1], dim=1)
        return self.conv(x)

# ---------- Rede principal ----------
class GraspNN_V1(nn.Module):
    """
    Entrada  : (B, 8, H, W)  – 3 RGB + 1 depth + 3 normal + 1 mask
    Saída    : (B, 1, H, W)  – logit por pixel
    """
    def __init__(self, in_channels=8, out_ch=1, base=32):
        super().__init__()
        self.inc   = double_conv(in_channels, base)           # 32
        self.down1 = Down(base, base*2)                 # 64
        self.down2 = Down(base*2, base*4)               # 128
        self.down3 = Down(base*4, base*8)               # 256
        self.up2   = Up(base*8, base*4)                 # 128
        self.up1   = Up(base*4, base*2)                 # 64
        self.up0   = Up(base*2, base)                   # 32
        self.outc  = nn.Conv2d(base, out_ch, 1)

    def forward(self, x):
        x0 = self.inc(x);           #print_cuda_memory()

        x1 = self.down1(x0);        #print_cuda_memory()

        x2 = self.down2(x1);        #print_cuda_memory()

        x3 = self.down3(x2);        #print_cuda_memory()

        x = self.up2(x3, x2);       #print_cuda_memory()

        x = self.up1(x,  x1);       #print_cuda_memory()

        x = self.up0(x,  x0);       #print_cuda_memory()
        return self.outc(x)             # logits; aplique sigmoid fora

#Visualize network architecture
summary(GraspNN_V1().to(device), input_size=(8,)+img_shape)

In [ ]:
model = GraspNN_V1().to(device)
batch = dataset[1]

x = torch.cat([
                batch["rgb"],
                batch["depth"],
                batch["normal"],
                batch["mask"]
                ],
                dim=0).to(device)

model(x.unsqueeze(dim=0)).shape

### Modelo 2

In [ ]:
# Rede extraida de artigo base

class Encoder(nn.Module):
    def __init__(self, in_channels: int = 8):      # ← agora 8
        super().__init__()
        self.layers = nn.Sequential(
            nn.Conv2d(in_channels, 8,  kernel_size=3, stride=3, padding=1),
            nn.ReLU(True),

            nn.Conv2d(8, 16, kernel_size=3, stride=2, padding=1),
            nn.ReLU(True),

            nn.Conv2d(16, 16, kernel_size=3, stride=2, padding=1),
            nn.ReLU(True),

            nn.Conv2d(16, 32, kernel_size=3, stride=1, padding=1),
            nn.ReLU(True),

            nn.Conv2d(32, 32, kernel_size=3, stride=1, padding=1),
            nn.ReLU(True),

            nn.Conv2d(32, 32, kernel_size=3, stride=1, padding=1),
            nn.ReLU(True),

            nn.Conv2d(32, 32, kernel_size=3, stride=1, padding=1),
            nn.ReLU(True),
        )

    def forward(self, x):
        return self.layers(x)          # (B, 32, 25, 25)


class Decoder(nn.Module):
    def __init__(self, out_channels: int = 1):     # ← agora 1
        super().__init__()

        self.conv_stack = nn.Sequential(
            nn.Conv2d(32, 32, kernel_size=3, padding=1),
            nn.ReLU(True),

            nn.Conv2d(32, 32, kernel_size=3, padding=1),
            nn.ReLU(True),

            nn.Conv2d(32, 32, kernel_size=3, padding=1),
            nn.ReLU(True),

            nn.Conv2d(32, 32, kernel_size=3, padding=1),
            nn.ReLU(True),
        )

        self.up1 = nn.ConvTranspose2d(32, 16, kernel_size=4, stride=2, padding=1)
        self.up2 = nn.ConvTranspose2d(16, 16, kernel_size=4, stride=2, padding=1)
        self.up3 = nn.ConvTranspose2d(16, 8,  kernel_size=3, stride=3, padding=(0,4))

        self.final = nn.Conv2d(8, out_channels, kernel_size=1)  # 8 → 1

    def forward(self, x):
        x = self.conv_stack(x)
        x = F.relu(self.up1(x))
        x = F.relu(self.up2(x))
        x = F.relu(self.up3(x))
        return self.final(x)           # (B, 1, 300, 300)


class GraspNN_V2(nn.Module):
    """
    Autoencoder completo. A saída `latent` (flatten 25×25×32) está
    disponível caso você queira usar como embedding para outra tarefa.
    """
    def __init__(self, in_channels: int = 8, out_channels: int = 1):
        super().__init__()
        self.encoder = Encoder(in_channels)
        self.decoder = Decoder(out_channels)

    def forward(self, x):
        latent = self.encoder(x)               # (B,32,25,25)
        out    = self.decoder(latent)
        return out#, latent.view(x.size(0), -1) # recon, embed

#Visualize network architecture
summary(GraspNN_V2().to(device), input_size=(8,)+img_shape)

In [ ]:
model = GraspNN_V2().to(device)
batch = dataset[1]

x = torch.cat([
                batch["rgb"],
                batch["depth"],
                batch["normal"],
                batch["mask"]
                ],
                dim=0).to(device)

model(x.unsqueeze(dim=0)).shape

### Modelo 3

In [ ]:
# Modelo "otimizado" pelo chat GPT a partir do modelo 1

from torchvision.models import mobilenet_v2

# ---------- bloco decoder (conv transposta + DW separável) ----------
class UpBlock(nn.Module):
    def __init__(self, in_up, in_skip, out_c, groups=1):
        super().__init__()
        self.up = nn.ConvTranspose2d(in_up, out_c, 2, stride=2, bias=False)
        self.conv = nn.Sequential(
            nn.Conv2d(out_c + in_skip, out_c, 3, padding=1, groups=groups, bias=False),
            nn.GroupNorm(8, out_c), nn.ReLU(inplace=True)
        )
    def forward(self, x_up, x_skip):
        x = self.up(x_up)
        x = torch.cat([x_skip, x], 1)
        return self.conv(x)


class GraspNN_V3(nn.Module):
    def __init__(self, in_channels=8, base=16):
        super().__init__()
        mb = mobilenet_v2(weights=None)
        # adapta 8 canais para 32
        mb.features[0][0] = nn.Conv2d(in_channels, 32, 3, 1, 1, bias=False)
        self.backbone = mb.features

        self.enc1 = self.backbone[:3]     # 24 ch
        self.enc2 = self.backbone[3:6]    # 32 ch
        self.enc3 = self.backbone[6:10]   # 64 ch   (skip)
        self.enc4 = self.backbone[10:]    # 1280 ch (bottleneck)

        self.up3 = UpBlock(1280, 64,  base*8)  # <-- canais certos
        self.up2 = UpBlock(base*8, 32, base*4)
        self.up1 = UpBlock(base*4, 24, base*2)
        self.up0 = nn.Sequential(
            nn.ConvTranspose2d(base*2, base, 2, stride=2, bias=False),
            nn.GroupNorm(8, base), nn.ReLU(inplace=True))
        self.outc = nn.Conv2d(base, 1, 1)

    def forward(self, x):
        s1 = self.enc1(x)
        s2 = self.enc2(s1)
        s3 = self.enc3(s2)
        x  = self.enc4(s3)

        x = self.up3(x, s3)
        x = self.up2(x, s2)
        x = self.up1(x, s1)
        x = self.up0(x)
        return self.outc(x)


### Modelo 4

In [ ]:
# Modelo extraído de artigo

class GraspNN_V4(nn.Module):
    def __init__(self, input_layer=1):
        super(GraspNN_V4, self).__init__()

        # Camadas de convolução (reduzem o tamanho)
        self.conv1 = nn.Conv2d(input_layer, 32, kernel_size=9, stride=1, padding=4)
        self.conv2 = nn.Conv2d(32, 16, kernel_size=5, stride=1, padding=2)
        self.conv3 = nn.Conv2d(16, 8, kernel_size=3, stride=1, padding=1)

        # Camadas deconvolutivas (aumentam o tamanho)
        self.conv4 = nn.ConvTranspose2d(8, 8, kernel_size=3, stride=1, padding=1, output_padding=0)
        self.conv5 = nn.ConvTranspose2d(8, 16, kernel_size=4, stride=1, padding=2, output_padding=0)
        self.conv6 = nn.ConvTranspose2d(16, 32, kernel_size=9, stride=1, padding=4, output_padding=0)

        # Ajuste de padding
        self.pad = nn.ConstantPad2d((0, 1, 0, 1), 0)

        # Camadas de saída
        self.pos_out = nn.Conv2d(32, 1, kernel_size=2)
        self.cos_out = nn.Conv2d(32, 1, kernel_size=2)
        self.sin_out = nn.Conv2d(32, 1, kernel_size=2)
        self.width_out = nn.Conv2d(32, 1, kernel_size=2)

        self.out = nn.Conv2d(32, 1, kernel_size=2)

    def forward(self, x):
        # Passagem pelas camadas de convolução
        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))
        #encoded = F.relu(self.conv3(x))

        # Passagem pelas camadas deconvolutivas
        #x = F.relu(self.conv4(encoded))
        #x = F.relu(self.conv5(x))
        x = F.relu(self.conv6(x))

        # Ajuste de padding
        x = self.pad(x)

        # Passagem pela camada de saída
        output = self.out(x)

        return output

#Visualize network architecture
from torchsummary import summary
summary(GraspNN_V4(1).to(device), input_size=(1,)+img_shape)

## Usando "depth", "segmentation", "rgb" e "normal_map"'

In [ ]:
model_version = GraspNN_V3()

torch.manual_seed(100)

model = model_version.to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-2, weight_decay=1e-4)

loss_fn = nn.MSELoss()

!nvidia-smi

In [ ]:
# Loop de Treinamento e Teste
num_epochs = 10

# Limiar
lim = 0.995

#Rastreio de valores de treinamento
epoch_count = []
loss_values = []
test_loss_values = []

for epoch in range(num_epochs):

    # Treinamento
    model.train()

    train_loss, train_acc = 0, 0
    for batch in tqdm(train_loader):
        x = torch.cat([
                       batch["rgb"],
                       batch["depth"],
                       batch["normal"],
                       batch["mask"]
                        ],
                      dim=1).to(device)
        y = batch["grasp"].unsqueeze(1).to(device)      # (B,1,H,W)

        optimizer.zero_grad()
        logits = model(x)
        loss   = loss_fn(logits, y)
        train_loss += loss
        train_acc += accuracy_fn(y_true=y,
                                 y_pred=(logits).type(torch.int)) # Go from logits -> pred labels

        loss.backward()
        optimizer.step()
        #print_cuda_memory()

    # Calculate loss and accuracy per epoch and print out what's happening
    train_loss /= len(train_loader)
    train_acc /= len(train_loader)

    print(f"\n\033[1;92m-----------------------  Epoch {epoch}:  -----------------------\033[0;0m")


    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()

    # Teste

    model.eval()

    test_loss, test_acc = 0, 0
    with torch.inference_mode():
        for batch in test_loader:
            # Send data to GPU
            x = torch.cat([
                       batch["rgb"],
                       batch["depth"],
                       batch["normal"],
                       batch["mask"]
                       ], dim=1).to(device)
            y = batch["grasp"].unsqueeze(1).to(device)      # (B,1,H,W)

            # 1. Forward pass
            test_pred = model(x)

            # 2. Calculate loss and accuracy
            test_loss += loss_fn(test_pred, y)
            test_acc += accuracy_fn(y_true=y,
                y_pred=(test_pred).type(torch.int) # Go from logits -> pred labels
            )

        if epoch % 1 == 0:
            epoch_count.append(epoch)
            loss_values.append(loss)
            test_loss_values.append(test_loss)

        # Adjust metrics and print out
        test_loss /= len(test_loader)
        test_acc /= len(test_loader)

    print(f"Train loss: {train_loss:.5f} | Train accuracy: {train_acc:.2f}%")
    print(f"Test loss: {test_loss:.5f} | Test accuracy: {test_acc:.2f}%")

    print(f"\033[1;92m----------------------------------------------------------\033[0;0m\n")

    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()

epoch_count_np = np.array(torch.tensor(epoch_count).numpy())
loss_values_np = np.array(torch.tensor(loss_values).numpy())
test_loss_values_np = np.array(torch.tensor(test_loss_values).numpy())

plt.figure(2, figsize=(10,7))
plt.plot(epoch_count_np, loss_values_np, label="Train loss")
plt.plot(epoch_count_np, test_loss_values_np, label="Test loss")
plt.title("Training and test loss curves")
plt.ylabel("Loss")
plt.xlabel("Epoch")
plt.legend()

In [ ]:
x.shape

In [ ]:
a = test_pred[0].squeeze().type(torch.int)
b = y[0].squeeze().type(torch.int)

In [ ]:
model_1 = model

## Usando apenas "depth"

In [ ]:
model_version = GraspNN_V4()

torch.manual_seed(42)

model = model_version.to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-4)

loss_fn = nn.MSELoss()

In [ ]:
mse = nn.MSELoss()

def loss_calculation(prediction, target):

    loss = mse(prediction, target)

    return loss

class Loss(torch.nn.modules.loss._Loss):

    def __init__(self):
        super(Loss, self).__init__(True)

    def forward(self, pos_pred, point, cos_pred, cos, sin_pred, sin, width_pred, width):
        pos_loss = loss_calculation(pos_pred, point)
        cos_loss = loss_calculation(cos_pred, cos)
        sin_loss = loss_calculation(sin_pred, sin)
        width_loss = loss_calculation(width_pred, width)
        total_loss = pos_loss + cos_loss + sin_loss + width_loss
        return pos_loss, cos_loss, sin_loss, width_loss, total_loss

In [ ]:
# Loop de Treinamento e Teste
num_epochs = 10

# Limiar
lim = 0.5

#Rastreio de valores de treinamento
epoch_count = []
loss_values = []
test_loss_values = []

for epoch in range(num_epochs):

    # Treinamento
    model.train()

    train_loss, train_acc = 0, 0
    for batch in tqdm(train_loader):
        x = batch["depth"].squeeze(dim=0).float().to(device)

        #x = torch.mul(batch["depth"], batch["mask"]).float().to(device)

        y = batch["grasp"].unsqueeze(dim=1).float().to(device)      # (B,1,H,W)

        optimizer.zero_grad()

        logits = model(x)

        loss   = loss_fn(logits, y)

        train_loss += loss
        train_acc += accuracy_fn(y_true=y,
                                 y_pred=(logits).type(torch.int) # Go from logits -> pred labels
        )
        loss.backward()
        optimizer.step()
        #print_cuda_memory()

    # Calculate loss and accuracy per epoch and print out what's happening
    train_loss /= len(train_loader)
    train_acc /= len(train_loader)

    print(f"\n\033[1;92m-----------------------  Epoch {epoch}:  -----------------------\033[0;0m")


    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()

    # Teste

    test_loss, test_acc = 0, 0
    model.eval()
    with torch.inference_mode():
        for batch in test_loader:
            # Send data to GPU

            x = batch["depth"].squeeze(dim=0).float().to(device)

            #x = torch.mul(batch["depth"], batch["mask"]).float().to(device)
            y = batch["grasp"].unsqueeze(dim=1).float().to(device)      # (B,1,H,W)

            # 1. Forward pass
            test_pred = model(x)

            # 2. Calculate loss and accuracy
            test_loss += loss_fn(test_pred, y)
            test_acc += accuracy_fn(y_true=y,
                                    y_pred=(test_pred).type(torch.int) # Go from logits -> pred labels
            )

        if epoch % 1 == 0:
            epoch_count.append(epoch)
            loss_values.append(loss)
            test_loss_values.append(test_loss)

        # Adjust metrics and print out
        test_loss /= len(test_loader)
        test_acc /= len(test_loader)

    print(f"Train loss: {train_loss:.5f} | Train accuracy: {train_acc:.2f}%")
    print(f"Test loss: {test_loss:.5f} | Test accuracy: {test_acc:.2f}%")

    print(f"\033[1;92m----------------------------------------------------------\033[0;0m\n")

    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()

# Plot do gráfico de percas

epoch_count_np = np.array(torch.tensor(epoch_count).numpy())
loss_values_np = np.array(torch.tensor(loss_values).numpy())
test_loss_values_np = np.array(torch.tensor(test_loss_values).numpy())

plt.figure(2, figsize=(10,7))
plt.plot(epoch_count_np, loss_values_np, label="Train loss")
plt.plot(epoch_count_np, test_loss_values_np, label="Test loss")
plt.title("Training and test loss curves")
plt.ylabel("Loss")
plt.xlabel("Epoch")
plt.legend()

In [ ]:
model_2 = model

# Resultados

## Plot das Previsões

In [ ]:
random_view_id = random.randint(0,views_count)

In [ ]:
view_rgb_t = dataset[random_view_id]["rgb"]
view_depth_t = dataset[random_view_id]["depth"]
view_normal_t = dataset[random_view_id]["normal"]
view_seg_t = dataset[random_view_id]["mask"]
view_target = dataset[random_view_id]["grasp"]

x = torch.cat([
    view_rgb_t,
    view_depth_t,
    view_normal_t,
    view_seg_t,
    ],dim=0).unsqueeze(dim=0).to(device)

plot_data = {
    "rgb" : view_rgb_t.permute(1, 2, 0).float().numpy(),
    "normal" : view_normal_t.permute(1, 2, 0).float().numpy(),
    "depth" : view_depth_t.permute(1, 2, 0).squeeze().numpy(),
    "mask" : view_seg_t.permute(1, 2, 0).squeeze().int().numpy()
}
target = view_target.int().numpy()

output_1 = plot_prediction(model=model_1,
                           input=x,
                           plot_data=plot_data,
                           target=target,
                           raw = True, lim = 0.99)

## Vista 97 ficou bala

In [ ]:
view_rgb_t = dataset[random_view_id]["rgb"]
view_depth_t = dataset[random_view_id]["depth"]
view_normal_t = dataset[random_view_id]["normal"]
view_seg_t = dataset[random_view_id]["mask"]
view_target = dataset[random_view_id]["grasp"]

x = view_depth_t.unsqueeze(dim=0).to(device)

plot_data = {
    "rgb" : view_rgb_t.permute(1, 2, 0).float().numpy(),
    "normal" : view_normal_t.permute(1, 2, 0).float().numpy(),
    "depth" : view_depth_t.permute(1, 2, 0).squeeze().numpy(),
    "mask" : view_seg_t.permute(1, 2, 0).squeeze().int().numpy()
}
target = view_target.int().numpy()

output_2 = plot_prediction(model=model_2,
                           input=x,
                           plot_data=plot_data,
                           target=target,
                           raw = True, lim = 0.99)

# Salvar Modelo

In [ ]:
# Nomeando o modelo
model = model_1

# Criando um diretório para os modelos
MODEL_PATH = Path("models")
MODEL_PATH.mkdir(parents=True, exist_ok=True)

# Lista para armazenar os números extraídos
numeros = []

# Regex para encontrar os números que seguem "nn_"
pattern = re.compile(r"nn-(\d+)_")

# Percorrer todos os arquivos no diretório
for arquivo in MODEL_PATH.iterdir():
    if arquivo.is_file():  # Verifica se é um arquivo (e não um diretório)
        # Verificar se o arquivo corresponde ao padrão
        match = pattern.match(arquivo.name)
        if match:
            # Adicionar o número extraído à lista
            numeros.append(int(match.group(1)))

# Obter o maior número, se houver
nn_count = max(numeros) if numeros else 0

MODEL_NAME = f"nn-{nn_count+1}_{datetime.now().strftime('%y-%m-%d')}_{model._get_name()}.pth"

In [ ]:
MODEL_NAME

In [ ]:
# Criando caminho para salvar modelo
MODEL_SAVE_PATH = MODEL_PATH / MODEL_NAME

MODEL_SAVE_PATH

# Salvando o modelo no formato state_dict
torch.save(obj=model.state_dict(),
           f=MODEL_SAVE_PATH)

# Validação no Gênesis

In [ ]:
import genesis as gs
import numpy as np
from pathlib import Path
import cv2
from scipy.spatial.transform import Rotation as R
from trimesh.transformations import quaternion_from_euler, euler_from_quaternion
import random

from d2nt_utils import *

In [ ]:
# Genesis Setup
gs.init(backend=gs.cuda, precision='32', theme='dark', eps=1e-12)
scene = gs.Scene(
    viewer_options=gs.options.ViewerOptions(camera_pos=(1, 1, 1), camera_lookat=(0, 0, 0.15), camera_fov=30, max_FPS=600),
    sim_options=gs.options.SimOptions(dt=0.001),
    show_viewer=True,
    show_FPS=False
)

plane = scene.add_entity(gs.morphs.Plane())
cylinder = scene.add_entity(
    gs.morphs.Cylinder(
        pos=[0.0, 0.0, 0.05],
        height=0.1,
        radius=0.03,
        collision=False,
        fixed=True
    )
)
spot_gripper = scene.add_entity(
    gs.morphs.URDF(
        file='urdf/spot_arm/urdf/open_gripper.urdf',
        euler=(90, 0, 0),
        pos=(-1, 0.0, 0.10),
        scale=1.0,
        merge_fixed_links=True,
        fixed=True
    )
)

camera = scene.add_entity(
    gs.morphs.Cylinder(
        pos=[0.0, 0.0, 1.0],
        height=0.1,
        radius=0.02,
        collision=False,
        fixed=True,
    )
)

cam = scene.add_camera(
    pos=[0.0, 0.0, 1.0],
    lookat = [0.0, 0.0, 0.15],
    res    = (img_shape[1],img_shape[0]),
    fov    = 30,
    GUI    = False,
)

# build
scene.build()

In [ ]:
# Define DOFs
hand_dof = np.arange(2)
finger_dof = np.array([1])

# Set PD control parameters
spot_gripper.set_dofs_kp(
    np.array([100]*2)
    )
spot_gripper.set_dofs_kv(
    np.array([1]*2)
    )
spot_gripper.set_dofs_force_range(
    np.array([-100]*2),
    np.array([100]*2)
    )

## Function

In [ ]:
# D2NT Functions
def compute_d2nt_normal(depth_path, normal_path, VERSION="d2nt_v3"):

    # Fake camera intrinsics (adjust based on Genesis camera if known)
    cam_fx, cam_fy, u0, v0 = 1280 / (2 * np.tan(np.deg2rad(30) / 2)), 960 / (2 * np.tan(np.deg2rad(30) / 2)), 640, 480

    # get ground truth normal [-1,1]
    normal_gt = get_normal_gt(normal_path)
    normal_gt = vector_normalization(normal_gt)
    h, w, _ = normal_gt.shape

    # get depth
    depth, mask = get_depth(depth_path, h, w)
    u_map = np.ones((h, 1)) * np.arange(1, w + 1) - u0
    v_map = np.arange(1, h + 1).reshape(h, 1) * np.ones((1, w)) - v0

    # get depth gradients
    if VERSION == 'd2nt_basic':
        Gu, Gv = get_filter(depth)
    else:
        Gu, Gv = get_DAG_filter(depth)

    # Depth to Normal Translation
    est_nx = Gu * cam_fx
    est_ny = Gv * cam_fy
    est_nz = -(depth + v_map * Gv + u_map * Gu)
    est_normal = cv2.merge((est_nx, est_ny, est_nz))

    # vector normalization
    est_normal = vector_normalization(est_normal)

    # MRF-based Normal Refinement
    if VERSION == 'd2nt_v3':
        est_normal = MRF_optim(depth, est_normal)

    return est_normal

def d2nt(depth, VERSION="d2nt_v3"):

    # Fake camera intrinsics (adjust based on Genesis camera if known)
    cam_fx, cam_fy, u0, v0 = 1280 / (2 * np.tan(np.deg2rad(30) / 2)), 960 / (2 * np.tan(np.deg2rad(30) / 2)), 640, 480
    h, w = depth.shape

    # get depth
    u_map = np.ones((h, 1)) * np.arange(1, w + 1) - u0
    v_map = np.arange(1, h + 1).reshape(h, 1) * np.ones((1, w)) - v0

    # get depth gradients
    if VERSION == 'd2nt_basic':
        Gu, Gv = get_filter(depth)
    else:
        Gu, Gv = get_DAG_filter(depth)

    # Depth to Normal Translation
    est_nx = Gu * cam_fx
    est_ny = Gv * cam_fy
    est_nz = -(depth + v_map * Gv + u_map * Gu)
    est_normal = cv2.merge((est_nx, est_ny, est_nz))

    # vector normalization
    est_normal = vector_normalization(est_normal)

    # MRF-based Normal Refinement
    if VERSION == 'd2nt_v3':
        est_normal = MRF_optim(depth, est_normal)

    return est_normal

In [ ]:
# Convert to world coordinates
def pixel_to_world(pixel_x, pixel_y, depth, cam_pos, cam_lookat, fov, img_width, img_height, world_up):

    cam_pos = np.array(cam_pos)
    cam_lookat = np.array(cam_lookat)

    # Calculate camera direction vector
    cam_dir = cam_lookat - cam_pos
    cam_dir = cam_dir / np.linalg.norm(cam_dir)

    # Calculate camera up vector (assuming world up is (0,0,1))
    world_up = np.array(world_up)
    cam_right = np.cross(cam_dir, world_up)
    cam_right = cam_right / np.linalg.norm(cam_right)
    cam_up = np.cross(cam_right, cam_dir)

    # Convert FOV to radians
    fov_rad = np.deg2rad(fov)

    # Calculate pixel coordinates in normalized device coordinates (-1 to 1)
    ndc_x = (2.0 * pixel_x / (img_width - 1)) - 1.0
    ndc_y = 1.0 - (2.0 * pixel_y / (img_height - 1))  # Flip y-axis

    # Calculate direction vector in camera space
    aspect_ratio = img_width / img_height
    tan_half_fov = np.tan(fov_rad / 2.0)

    cam_space_x = ndc_x * tan_half_fov * aspect_ratio
    cam_space_y = ndc_y * tan_half_fov
    cam_space_z = -1.0  # Negative z is forward in camera space

    # Create direction vector in camera space
    cam_space_dir = np.array([cam_space_x, cam_space_y, cam_space_z])
    cam_space_dir = cam_space_dir / np.linalg.norm(cam_space_dir)

    # Convert to world space direction
    world_dir = (cam_right * cam_space_dir[0] +
                cam_up * cam_space_dir[1] +
                -cam_dir * cam_space_dir[2])
    world_dir = world_dir / np.linalg.norm(world_dir)

    # Calculate world position
    world_pos = cam_pos + world_dir * depth

    return world_pos

In [ ]:
# Rotation

def normalize(v):
    return v / np.linalg.norm(v)

def rotate_vector(normal, cam_pos, cam_lookat, up):
    # Vetor direção da câmera (câmera olha de cam_pos para cam_lookat)
    z_cam = normalize(np.array(cam_pos) - np.array(cam_lookat))

    # Vetor "para cima" definido pelo usuário
    up = np.array(up)

    # Vetor X da câmera, perpendicular ao Z (produto vetorial de up e z_cam)
    x_cam = normalize(np.cross(up, z_cam))

    # Vetor Y da câmera, perpendicular ao plano XZ (produto vetorial de z_cam e x_cam)
    y_cam = np.cross(z_cam, x_cam)

    rotation_matrix = np.column_stack([x_cam, y_cam, z_cam])

    # Aplicando a transformação nos vetores normais (multiplicação matricial)
    transformed_normals = np.dot(normal, rotation_matrix.T)  # Transposta de R para a transformação correta

    return transformed_normals


In [ ]:
#Vector Conversion
def vector_to_euler(vec):
    vec = np.array(vec) / np.linalg.norm(vec)  # Normalize the vector
    reference = np.array([1, 0, 0])  # Reference vector

    if np.allclose(vec, reference):
        return np.array([0.0, 0.0, 0.0])

    axis = np.cross(reference, vec)
    angle = np.arccos(np.dot(reference, vec))

    if np.linalg.norm(axis) < 1e-6:
        axis = np.array([0, 0, 1])
    else:
        axis = axis / np.linalg.norm(axis)

    rotation = R.from_rotvec(angle * axis)
    print(rotation)
    euler_angles = rotation.as_euler('xyz')

    return euler_angles

def quaternion_from_vectors(v_from, v_to):
    # Normaliza:
    v_from = v_from / np.linalg.norm(v_from)
    v_to   = v_to   / np.linalg.norm(v_to)

    # dot product -> cosseno do ângulo:
    dot_val = np.dot(v_from, v_to)
    # clamp para evitar problema se cair fora de [-1, 1] por ruído
    dot_val = np.clip(dot_val, -1.0, 1.0)

    theta = np.arccos(dot_val)  # ângulo escalar

    # Se o ângulo for pequeno, retorna identidade
    if abs(theta) < 1e-8:
        return np.array([1, 0, 0, 0], dtype=float)

    # Se o ângulo estiver perto de 180º:
    if abs(theta - np.pi) < 1e-8:
        # Precisamos de um eixo ortogonal a v_from
        # Pega um qualquer ortogonal a v_from
        # Exemplo: eixos canônicos e escolhe o que for menos paralelo
        axis_candidates = [np.array([1,0,0], dtype=float),
                           np.array([0,1,0], dtype=float),
                           np.array([0,0,1], dtype=float)]
        best_axis = None
        best_dot = 1.0
        for c in axis_candidates:
            d_ = abs(np.dot(c, v_from))
            if d_ < best_dot:
                best_dot = d_
                best_axis = c
        axis = np.cross(v_from, best_axis)
        axis /= np.linalg.norm(axis)
    else:
        # Eixo = cross(v_from, v_to)
        axis = np.cross(v_from, v_to)
        axis /= np.linalg.norm(axis)

    half_angle = theta / 2
    w = np.cos(half_angle)
    sin_h = np.sin(half_angle)
    x, y, z = axis * sin_h
    return np.array([w, x, y, z], dtype=float)

## Simulação de Deploy

### Random Camera View

In [ ]:
#------------Cam Static Values-------------#
cam_lookat_0 = [0.0, 0.0, 0.05]
cam_lookat = cam_lookat_0
fov = 30
img_width, img_height = img_shape[1],img_shape[0]
#------------------------------------------#

#------------Cilinder-------------#
cylinder.set_pos(cam_lookat)
cylinder.set_quat(quaternion_from_euler(0,0,0,'ryxz'))
#---------------------------------#

dv=[0,0,0]

smp = 0

In [ ]:
# Move Scene
dx = 0.0
dy = 0.0
dz = 0.0
dv=[dx,dy,dz]

print(f"Deslocamento: \033[1;92m{dv}\033[0m")

In [ ]:
# Random View
cam_lookat = cam_lookat_0
up = [0, 0, 1]

#randomize camera position
rand_cam_x = random.uniform(-2.5, 2.5)
rand_cam_y = random.uniform(-2.5, 2.5)
rand_cam_z = random.uniform(0.005, 2.0)
cam_pos=[rand_cam_x,rand_cam_y,rand_cam_z]

for i in range(3):
    cam_pos[i]=cam_pos[i]+dv[i]
    cam_lookat[i]=cam_lookat[i]+dv[i]

#get camera info
cam.set_pose(
    pos = (cam_pos[0] + cam_lookat[0],cam_pos[1] + cam_lookat[1],cam_pos[2] + cam_lookat[2]),
    lookat = cam_lookat,
)
cylinder.set_pos(cam_lookat)
camera.set_pos(cam_pos)
camera.set_quat(quaternion_from_vectors(cam_lookat,cam_pos))

for i in range(5):
    scene.step()
    render_output = cam.render(rgb=True, depth=True, segmentation=True, normal=True, colorize_seg=False, )

rgb, depth, seg, normal_map = render_output

normal_map = normalize(normal_map)

# Compute Normal Map using D2NT
normal_map_d2nt = d2nt(depth)

### NN

In [ ]:
# Carregando o modelo salvo
MODEL_PATH = Path("models/nn-2_25-04-28_GraspNN_V3.pth")
loaded_model = GraspNN_V3() # cria um objeto com o mesmo tipo dos dados anteriores

loaded_model.load_state_dict(torch.load(f=MODEL_PATH))
loaded_model.to(device)
list(loaded_model.parameters())

In [ ]:
rgb_, depth_, normal_map_, seg, _= preprocess_data(rgb, depth, normal_map_d2nt, seg, target=np.zeros(img_shape))

view_rgb_t = torch.from_numpy(np.copy(rgb_)).permute(2,0,1).float()
view_depth_t = torch.from_numpy(np.copy(depth_)).unsqueeze(dim=2).permute(2,0,1).float()
view_normal_t = torch.from_numpy(np.copy(normal_map_)).permute(2,0,1).float()
view_seg_t = torch.from_numpy(np.copy(seg)).unsqueeze(dim=2).permute(2,0,1).float()

x_1 = torch.cat([
    view_rgb_t,
    view_depth_t,
    view_normal_t,
    view_seg_t,
    ],dim=0).unsqueeze(dim=0).to(device)

x_2 = view_depth_t.unsqueeze(dim=0).to(device)

with torch.inference_mode():
    output = loaded_model(x_1).squeeze().cpu()

grasp_y, grasp_x = np.unravel_index(np.argmax(output*seg), output.shape)

### Performing Grasp

In [ ]:
# Select a random cylinder pixel
specific_point_depth = depth[grasp_y, grasp_x]
specific_point_normal = -normal_map_d2nt[grasp_y, grasp_x]*0.35  # D2NT normal (already [-1, 1]) com 30 cm de tamanho

specific_point_normal = rotate_vector(specific_point_normal, cam_pos, cam_lookat, up) #vetor com 30 cm de tamanho

grasp_point = pixel_to_world(grasp_x, grasp_y, specific_point_depth, cam_pos, cam_lookat, fov, img_width, img_height, up)
#orientation_vec =  normalize(np.array(cam_lookat)-np.array(cam_pos))
#grasp_point = cam_pos + orientation_vec*specific_point_depth

position = grasp_point + specific_point_normal

euler = vector_to_euler(-1*specific_point_normal); euler[0] = 3*3.1415/2 - 0.31415

quartenion = quaternion_from_euler(*euler)

print(f"Grasp pixel coordinate: \033[1;92m({grasp_y}, {grasp_x})\033[0m")

print(f"\033[1;92m{"_______________________________________________________"}\033[0m")

print(f"Orientation: \033[1;92m{euler_from_quaternion(quartenion)}\033[0m")

print(f"\033[1;92m{"_______________________________________________________"}\033[0m")

print(f"Grasp Point: \033[1;92m{grasp_point}\033[0m")

print(f"Normal specific Point: \033[1;92m{specific_point_normal}\033[0m")

print(f"Gripper Graspping Point: \033[1;92m{position}\033[0m")

print(f"\033[1;92m{"_______________________________________________________"}\033[0m")

for _ in range(1):
    scene.step()

spot_gripper.set_quat(quartenion)
spot_gripper.set_pos(position)

print(f"Gripper Position: {spot_gripper.get_pos()}")

for _ in range(1):
    scene.step()  # Offset along normal
    rgb_post_alignment = cam.render(rgb=True)

In [ ]:
a = euler_from_quaternion(quartenion)
b = np.array([a[0],a[1],a[2]])*360/(2*3.1415)
b

In [ ]:
np.cross(specific_point_normal,[0,0,1])

In [ ]:
euler*360/(2*3.1415)

### Results

In [ ]:
plot_data = {
    "rgb" : rgb.astype(int),
    "normal" : normal_map_d2nt,
    "depth" : depth,
    "mask" : seg
}

output = plot_prediction(model=loaded_model,
                           input=x_1,
                           plot_data=plot_data,
                           raw = True, lim = 0.99)

print(f"Pixel para grasp: {grasp_y, grasp_x}")
grasp_point_map = output
grasp_point_map[grasp_y, grasp_x] = 9.99

fig, ax = plt.subplots(1, 2, figsize=(16,8))

plot1 = ax[0]
plot2 = ax[1]

plot1.imshow(grasp_point_map, cmap="gray")
plot1.set_title("Grasp Point")
plot1.axis(False);

plot2.imshow(rgb_post_alignment[0])
plot2.set_title("Post Alignment")
plot2.axis(False);